In [1]:
# Importing libraries and Setup
import os
from dotenv import load_dotenv
import requests
from typing import List, Literal
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

In [2]:
# Loading the API keys
if load_dotenv(override=True):
    print(f"API are loaded!")
    
# LLM API keys
openai_api_key = os.getenv("OPENAI_API_KEY")

# Email Configurations
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

# Pushover notification Configuration
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

API are loaded!


In [3]:
# Test the LLM calls
MODEL_NAME = 'gpt-4o-mini'

test_message = "Hi There!"
llm = ChatOpenAI(model=MODEL_NAME, api_key=openai_api_key)
llm.invoke(test_message).content

'Hello! How can I assist you today?'

In [4]:
# function to send push notification
@tool
def push(message: str) -> None:
    """Send push notification to the user"""
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

# create a tool list
tools = [push]

# add the tools with LLM instance
llm_with_tool = llm.bind_tools(tools)

In [10]:
# Building agent with Langgraph
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import tools_condition

In [6]:
from typing import TypedDict, Annotated
import operator

class EmailState(TypedDict):
    user_request: str
    emails: Annotated[List[str], operator.add]
    messages: list

In [12]:
# Instruction for sales agents
sales_intro = """
You are a sales agent working for ComplAI,
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails.
"""

instructions1 = sales_intro + "Your email style is professional, serious, with gravitas and credibility"
instructions2 = sales_intro + "Your email style is witty, engagging, and humorous"
instructions3 = sales_intro + "Your email style is concise, to the point, in the style of a busy senior executive."


selector_instruction = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give an explanation. Send the best email with push tool you have to the user.
"""

def sales_agent1(state: EmailState):
    messages = [SystemMessage(content=instructions1), *state['user_request']]
    response = llm.invoke(messages)
    return {"emails": [response.content]}

def sales_agent2(state: EmailState):
    messages = [SystemMessage(content=instructions2), *state['user_request']]
    response = llm.invoke(messages)
    return {"emails": [response.content]}    

def sales_agent3(state: EmailState):
    messages = [SystemMessage(content=instructions3), *state['user_request']]
    response = llm.invoke(messages)
    return {"emails": [response.content]}

def sales_picker(state: EmailState):
    messages = [SystemMessage(content=selector_instruction), *state['emails']]
    response = llm_with_tool.invoke(messages)
    return {"messages": [response]}

# Initialize the workflow
workflow = StateGraph(EmailState)

# creating the nodes
# Sales nodes
workflow.add_node('sales_agent1', sales_agent1)
workflow.add_node('sales_agent2', sales_agent2)
workflow.add_node('sales_agent3', sales_agent3)

# Tool node
tool_node = ToolNode(tools)
workflow.add_node('tools', tool_node)

# Selector node
workflow.add_node('selector', sales_picker)

# creating the flow
workflow.add_edge(START, 'sales_agent1')
workflow.add_edge(START, 'sales_agent2')
workflow.add_edge(START, 'sales_agent3')

workflow.add_edge('sales_agent1', 'selector')
workflow.add_edge('sales_agent2', 'selector')
workflow.add_edge('sales_agent3', 'selector')

workflow.add_conditional_edges('selector', tools_condition, {'tools': 'tools', END:END})
workflow.add_edge('tools', END)

sales_agent = workflow.compile()
sales_agent

user_message = "Write a cold sales email"
response = sales_agent.invoke({'user_request': user_message})